# Quantum Circuit Classifier Training Notebook
This notebook loads extracted circuit features, trains a classifier,
and evaluates its performance using cross-validation, feature importance, and confusion matrix.

In [5]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import joblib

In [6]:
# Load data
df = pd.read_csv('data/processed_features.csv')
df.head()

,filename,num_qubits,num_clbits,depth,total_gates,entangling_gates,gate_rz,gate_sx,gate_cx,gate_measure,gate_x,gate_barrier,label
0,grover_n2_transpiled.qasm,2,2,12,17,2,9,4,2,2,NaN,NaN,grover
1,wstate_n3_transpiled.qasm,3,3,24,38,9,17,7,9,3,2.0,NaN,wstate
2,ghz_state_n23_transpiled.qasm,23,46,26,49,22,2,1,22,23,NaN,1.0,ghz
3,vqe_uccsd_n8_transpiled.qasm,8,8,7657,9684,5284,3117,1226,5284,8,49.0,NaN,vqe
4,qaoa_n3_transpiled.qasm,3,3,17,35,6,17,9,6,3,NaN,NaN,qaoa


In [7]:
# Prepare features and labels
X = df.drop(columns=['filename', 'label'])
y = df['label']

In [9]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
X_train

,num_qubits,num_clbits,depth,total_gates,entangling_gates,gate_rz,gate_sx,gate_cx,gate_measure,gate_x,gate_barrier
8,2,2,214,308,42,164,100,42,2,NaN,NaN
5,6,6,150,384,54,196,124,54,6,4.0,NaN
2,23,46,26,49,22,2,1,22,23,NaN,1.0
1,3,3,24,38,9,17,7,9,3,2.0,NaN
12,8,8,246,1424,192,760,464,192,8,NaN,NaN
4,3,3,17,35,6,17,9,6,3,NaN,NaN
7,9,6,95,162,43,89,18,43,6,3.0,3.0
10,4,4,27,49,12,26,4,12,4,2.0,1.0
3,8,8,7657,9684,5284,3117,1226,5284,8,49.0,NaN
6,25,1,110,343,96,172,74,96,1,NaN,NaN


In [10]:
# Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.0


In [ ]:
# Cross-validation
cv_score = cross_val_score(clf, X, y, cv=5).mean()
print("Cross-validation accuracy:", cv_score)

In [ ]:
# Feature importance
importances = clf.feature_importances_
feat_names = X.columns
sns.barplot(x=importances, y=feat_names)
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Optional: Hyperparameter tuning
# Uncomment to use
# param_grid = {
#     'n_estimators': [50, 100, 150],
#     'max_depth': [None, 10, 20]
# }
# grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=5)
# grid.fit(X, y)
# print("Best Parameters:", grid.best_params_)
# clf = grid.best_estimator_

In [ ]:
# Save model
joblib.dump(clf, 'classifiers/quantum_classifier.pkl')